# Hierarchical Agent Structures

**Multigen SDK — Notebook 24**

---

## What are hierarchical agent structures?

In real-world multi-agent systems, agents are rarely peers.  They form **hierarchies**:

- A **coordinator** assigns work and aggregates results.
- **Workers** execute specialised tasks.
- **Critics** evaluate outputs and request revisions.
- **Sub-workers** handle lower-level subtasks delegated by workers.

The Multigen SDK models these relationships with:

| Class | Purpose |
|-------|---------|
| `AgentRole` | Enum of standard roles: `COORDINATOR`, `WORKER`, `CRITIC`, `PLANNER`, `EXECUTOR` |
| `AgentSpec` | Descriptor for a single agent: role, capabilities, version |
| `AgentGroup` | Named collection of agents; supports add/remove/filter by role |
| `AgentHierarchy` | Tree of `AgentSpec` nodes; tracks parent-child relationships |
| `TypedStep` | A `Step` with declared input/output JSON schemas and validation |
| `StepKind` | Enum that annotates step semantics: `TRANSFORM`, `FILTER`, `AGGREGATE`, `BRANCH`, `LOOP` |
| `HierarchicalPipeline` | A pipeline that mirrors an agent hierarchy — supports embed, flatten, describe |

## What this notebook covers

| Section | Topic |
|---------|-------|
| 1 | Setup and imports |
| 2 | AgentSpec — defining agents |
| 3 | AgentGroup — managing collections |
| 4 | AgentHierarchy — building multi-level trees |
| 5 | Navigating the hierarchy |
| 6 | AgentHierarchy.to_dict() and round-trip |
| 7 | TypedStep — steps with schema validation |
| 8 | TypedStep validation — valid and invalid inputs |
| 9 | HierarchicalPipeline — nesting, flatten, describe |
| 10 | Running a HierarchicalPipeline |
| 11 | StepKind taxonomy |
| 12 | Real-world research pipeline hierarchy |

All code is pure in-memory.  No external APIs required.


## Section 1 — Setup and Imports

We import the hierarchy and typed-step components from the `multigen` package.

- **`AgentRole`** — enum of standard roles.
- **`StepKind`** — enum for step semantics.
- **`AgentSpec`** — a single agent descriptor (name, role, capabilities, version).
- **`AgentGroup`** — a mutable collection of `AgentSpec` objects.
- **`AgentHierarchy`** — tree structure built from `AgentSpec` nodes.
- **`TypedStep`** — step with `input_schema` and `output_schema` dicts (JSON Schema).
- **`HierarchicalPipeline`** — pipeline whose structure mirrors an `AgentHierarchy`.


In [ ]:
import asyncio
import json
import sys

sys.path.insert(0, '../sdk')

from multigen.hierarchy import (
    AgentRole,
    StepKind,
    AgentSpec,
    AgentGroup,
    AgentHierarchy,
    TypedStep,
    HierarchicalPipeline,
)

print('AgentRole           :', AgentRole)
print('StepKind            :', StepKind)
print('AgentSpec           :', AgentSpec)
print('AgentGroup          :', AgentGroup)
print('AgentHierarchy      :', AgentHierarchy)
print('TypedStep           :', TypedStep)
print('HierarchicalPipeline:', HierarchicalPipeline)
print()

print('AgentRole members   :', [r.value for r in AgentRole])
print('StepKind members    :', [k.value for k in StepKind])
print()
print('All imports successful.')


## Section 2 — AgentSpec — Defining Agents

`AgentSpec` is a lightweight descriptor — it describes an agent but does not execute
any logic.  Think of it as a **schema record** in a registry.

Constructor:

```python
AgentSpec(
    name: str,
    role: AgentRole,
    capabilities: list[str] = [],
    version: str = "1.0.0",
    metadata: dict = {},
)
```

Key fields:

| Field | Type | Description |
|-------|------|-------------|
| `name` | str | Unique agent identifier |
| `role` | AgentRole | Role enum value |
| `capabilities` | list[str] | Tags describing what this agent can do |
| `version` | str | Semantic version string |
| `metadata` | dict | Arbitrary extra data |


In [ ]:
# Define several AgentSpec instances covering different roles

coordinator = AgentSpec(
    name='main_coordinator',
    role=AgentRole.COORDINATOR,
    capabilities=['task_dispatch', 'result_aggregation', 'routing'],
    version='2.1.0',
    metadata={'description': 'Top-level orchestrator for the research pipeline'},
)

planner = AgentSpec(
    name='research_planner',
    role=AgentRole.PLANNER,
    capabilities=['query_decomposition', 'priority_ordering'],
    version='1.0.0',
)

researcher = AgentSpec(
    name='web_researcher',
    role=AgentRole.WORKER,
    capabilities=['web_search', 'document_retrieval', 'summarisation'],
    version='1.3.0',
)

analyst = AgentSpec(
    name='data_analyst',
    role=AgentRole.WORKER,
    capabilities=['statistical_analysis', 'chart_generation', 'insight_extraction'],
    version='1.1.0',
)

writer = AgentSpec(
    name='report_writer',
    role=AgentRole.WORKER,
    capabilities=['markdown_generation', 'citation_formatting'],
    version='1.0.0',
)

critic = AgentSpec(
    name='quality_critic',
    role=AgentRole.CRITIC,
    capabilities=['fact_checking', 'style_review', 'completeness_scoring'],
    version='1.5.0',
)

all_specs = [coordinator, planner, researcher, analyst, writer, critic]

print('AgentSpec objects:')
for spec in all_specs:
    print(f'  {spec.name!r:<22}  role={spec.role.value!r:<14}  version={spec.version!r}  caps={spec.capabilities}')


## Section 3 — AgentGroup — Managing Collections

`AgentGroup(name)` is a mutable container for `AgentSpec` objects.

Key methods:

| Method | Description |
|--------|-------------|
| `add_agent(spec)` | Add an `AgentSpec` to the group |
| `remove_agent(name)` | Remove by agent name |
| `get_agent(name)` | Retrieve a specific spec |
| `agents_by_role(role)` | Return all specs matching a given `AgentRole` |
| `list_agents()` | Return all specs |
| `__len__()` | Number of agents in the group |


In [ ]:
group = AgentGroup('research_team')

# Populate the group
for spec in all_specs:
    group.add_agent(spec)

print(f'Group name    : {group.name}')
print(f'Total agents  : {len(group)}')
print()

# List all agents
print('All agents:')
for spec in group.list_agents():
    print(f'  {spec.name!r:<22}  role={spec.role.value!r}')
print()

# Filter by role
workers = group.agents_by_role(AgentRole.WORKER)
print(f'Workers ({len(workers)}):')
for spec in workers:
    print(f'  {spec.name!r}  capabilities={spec.capabilities}')
print()

# Retrieve a specific agent
retrieved = group.get_agent('quality_critic')
print(f'Retrieved agent  : {retrieved.name!r}  role={retrieved.role.value!r}')
print()

# Remove an agent
group.remove_agent('quality_critic')
print(f'After removing quality_critic: {len(group)} agents remain')
print('Still present:', [s.name for s in group.list_agents()])

# Re-add it for later sections
group.add_agent(critic)
print(f'Re-added critic: {len(group)} agents total')


## Section 4 — AgentHierarchy — Building a Multi-Level Tree

`AgentHierarchy` stores a tree of `AgentSpec` nodes.  Relationships are
parent-child: a coordinator can have workers as children, who can have sub-workers
as their children.

Key methods:

| Method | Description |
|--------|-------------|
| `set_root(spec)` | Set the root node |
| `add_child(parent_name, spec)` | Attach a spec as a child of the named parent |
| `visualize()` | Print an ASCII tree representation |
| `depth()` | Return the maximum depth of the tree |


In [ ]:
# Build a 3-level hierarchy:
# Level 0: coordinator
# Level 1: planner, researcher, analyst, writer
# Level 2: sub-workers under researcher and analyst

sub_researcher = AgentSpec(
    name='arxiv_specialist',
    role=AgentRole.WORKER,
    capabilities=['arxiv_search', 'paper_summarisation'],
    version='1.0.0',
)

sub_analyst = AgentSpec(
    name='stats_specialist',
    role=AgentRole.WORKER,
    capabilities=['regression_analysis', 'hypothesis_testing'],
    version='1.0.0',
)

hierarchy = AgentHierarchy()
hierarchy.set_root(coordinator)

# Level 1 — direct children of coordinator
for spec in [planner, researcher, analyst, writer, critic]:
    hierarchy.add_child('main_coordinator', spec)

# Level 2 — sub-workers
hierarchy.add_child('web_researcher', sub_researcher)
hierarchy.add_child('data_analyst',   sub_analyst)

print(f'Hierarchy depth: {hierarchy.depth()}')
print()
print('ASCII tree:')
hierarchy.visualize()


## Section 5 — Navigating the Hierarchy

`AgentHierarchy` exposes several navigation methods:

| Method | Returns | Description |
|--------|---------|-------------|
| `find_agent(name)` | `AgentSpec\|None` | Locate a node anywhere in the tree |
| `agents_by_role(role)` | `list[AgentSpec]` | All nodes with a given role |
| `path_to(name)` | `list[str]` | Ancestor names from root to target |
| `children_of(name)` | `list[AgentSpec]` | Direct children of a node |
| `parent_of(name)` | `AgentSpec\|None` | Parent node |
| `depth_of(name)` | `int` | Depth of a specific node (root = 0) |


In [ ]:
# find_agent
found = hierarchy.find_agent('stats_specialist')
print(f'find_agent("stats_specialist")  : {found.name!r}  role={found.role.value!r}')

not_found = hierarchy.find_agent('nonexistent_agent')
print(f'find_agent("nonexistent_agent") : {not_found}')
print()

# agents_by_role
all_workers = hierarchy.agents_by_role(AgentRole.WORKER)
print(f'All WORKER nodes in hierarchy ({len(all_workers)}):')
for w in all_workers:
    print(f'  {w.name!r}')
print()

# path_to
path = hierarchy.path_to('stats_specialist')
print(f'Path to "stats_specialist" : {path}')

path_root = hierarchy.path_to('main_coordinator')
print(f'Path to root               : {path_root}')
print()

# children_of
children = hierarchy.children_of('main_coordinator')
print(f'Children of main_coordinator ({len(children)}):')
for c in children:
    print(f'  {c.name!r}  role={c.role.value!r}')
print()

# parent_of
parent = hierarchy.parent_of('web_researcher')
print(f'Parent of web_researcher   : {parent.name!r}')

root_parent = hierarchy.parent_of('main_coordinator')
print(f'Parent of root             : {root_parent}')
print()

# depth_of
for name in ['main_coordinator', 'web_researcher', 'arxiv_specialist']:
    d = hierarchy.depth_of(name)
    print(f'depth_of({name!r:<22}) : {d}')


## Section 6 — AgentHierarchy.to_dict() and Round-Trip

`hierarchy.to_dict()` serialises the full tree to a JSON-compatible nested dict.
`AgentHierarchy.from_dict(d)` reconstructs a hierarchy from that dict.

This enables:

- Persisting the hierarchy to a file or database.
- Sending it over a network as JSON.
- Diffing two hierarchy versions.
- Logging the team composition alongside a workflow trace.


In [ ]:
serialised = hierarchy.to_dict()

print('Serialised hierarchy (JSON):')
print(json.dumps(serialised, indent=2))
print()

# Round-trip: reconstruct from dict
reconstructed = AgentHierarchy.from_dict(serialised)

print(f'Reconstructed depth         : {reconstructed.depth()}')
print(f'Reconstructed root name     : {reconstructed.find_agent("main_coordinator").name!r}')
print(f'Workers in reconstruction   : {[w.name for w in reconstructed.agents_by_role(AgentRole.WORKER)]}')
print()

# Verify the reconstructed hierarchy matches the original
original_names = sorted(
    node.name for node in hierarchy.agents_by_role(AgentRole.WORKER)
)
rebuilt_names = sorted(
    node.name for node in reconstructed.agents_by_role(AgentRole.WORKER)
)
assert original_names == rebuilt_names, 'Round-trip mismatch!'
print('Round-trip verification passed: original and reconstructed hierarchies match.')


In [ ]:
from multigen.hierarchy import TypedStep, StepKind, ValidationError

# Define input/output schemas as plain JSON Schema dicts

TEXT_INPUT_SCHEMA = {
    'type': 'object',
    'required': ['text'],
    'properties': {
        'text':     {'type': 'string', 'minLength': 1},
        'max_words': {'type': 'integer', 'minimum': 1},
    },
    'additionalProperties': True,
}

KEYWORDS_OUTPUT_SCHEMA = {
    'type': 'object',
    'required': ['keywords', 'word_count'],
    'properties': {
        'keywords':   {'type': 'array', 'items': {'type': 'string'}},
        'word_count': {'type': 'integer', 'minimum': 0},
    },
}

async def extract_keywords(input_data: dict) -> dict:
    text      = input_data['text']
    max_words = input_data.get('max_words', 10)
    words     = [w.lower().strip('.,!?') for w in text.split() if len(w) > 3][:max_words]
    return {
        'keywords':   words,
        'word_count': len(text.split()),
    }

typed_keyword_step = TypedStep(
    name='extract_keywords',
    fn=extract_keywords,
    input_schema=TEXT_INPUT_SCHEMA,
    output_schema=KEYWORDS_OUTPUT_SCHEMA,
    kind=StepKind.TRANSFORM,
)

print(f'TypedStep name        : {typed_keyword_step.name}')
print(f'TypedStep kind        : {typed_keyword_step.kind.value}')
print(f'TypedStep input keys  : {list(typed_keyword_step.input_schema.get("properties", {}).keys())}')
print(f'TypedStep output keys : {list(typed_keyword_step.output_schema.get("properties", {}).keys())}')
print()

# Validate input without running — returns True or raises
is_valid = typed_keyword_step.validate_input({'text': 'Hello world from Multigen SDK', 'max_words': 5})
print(f'validate_input (valid input) : {is_valid}')
print()

# Run the step
result = await typed_keyword_step.run({'text': 'Multigen SDK enables composable pipelines for multi-agent workflows', 'max_words': 5})
print(f'StepResult success    : {result.success}')
print(f'StepResult output     : {result.output}')


## Section 8 — TypedStep Validation — Valid and Invalid Inputs

`TypedStep.validate_input(input_dict)` checks the input against `input_schema`.

- If the input is **valid**: returns `True`.
- If the input is **invalid**: raises `ValidationError` with a descriptive message.

This lets you detect contract violations early — before any expensive LLM calls.

The same pattern applies to output validation via `validate_output(output_dict)`.


In [ ]:
print('=== Validation examples ===')
print()

# --- Valid input ---
valid_cases = [
    {'text': 'short text'},
    {'text': 'longer text with many words', 'max_words': 3},
]

for case in valid_cases:
    result = typed_keyword_step.validate_input(case)
    print(f'validate_input({case}) -> {result}')
print()

# --- Invalid inputs ---
invalid_cases = [
    ({}, 'missing required field "text"'),
    ({'text': ''}, 'empty string violates minLength: 1'),
    ({'text': 'hello', 'max_words': 0}, 'max_words violates minimum: 1'),
    ({'text': 42}, 'text must be a string, not integer'),
]

for inp, description in invalid_cases:
    try:
        typed_keyword_step.validate_input(inp)
        print(f'  UNEXPECTED PASS for {inp!r}')
    except ValidationError as e:
        print(f'  ValidationError ({description}):')
        print(f'    {e}')
    print()

# --- run() also validates before executing ---
print('run() with invalid input raises ValidationError before calling fn:')
try:
    await typed_keyword_step.run({'text': 123})
    print('  UNEXPECTED: run did not raise')
except ValidationError as e:
    print(f'  Caught ValidationError: {e}')
print()

# Confirm output validation
print('Output schema validation:')
valid_output   = {'keywords': ['hello', 'world'], 'word_count': 5}
invalid_output = {'keywords': 'not-a-list', 'word_count': 5}

print(f'  validate_output (valid)   : {typed_keyword_step.validate_output(valid_output)}')
try:
    typed_keyword_step.validate_output(invalid_output)
    print('  UNEXPECTED: no error raised')
except ValidationError as e:
    print(f'  validate_output (invalid) raises ValidationError: {e}')


## Section 9 — HierarchicalPipeline — Nesting, Flatten, Describe

`HierarchicalPipeline` organises `TypedStep` (or plain `Step`) objects into a tree
that mirrors an `AgentHierarchy`.  Each node in the pipeline corresponds to a node
in the hierarchy.

Key methods:

| Method | Description |
|--------|-------------|
| `embed(hierarchy, step_map)` | Populate the pipeline from an `AgentHierarchy` and a dict mapping agent names to steps |
| `flatten()` | Return a list of all steps in depth-first order |
| `describe()` | Return a nested dict describing the pipeline structure |
| `get_step(agent_name)` | Retrieve the step associated with an agent |


In [ ]:
from multigen.hierarchy import HierarchicalPipeline
from multigen.steps import Step

# Define simple async functions for each agent in our hierarchy
async def run_coordinator(inp): return {'coordinator_done': True, 'task': inp.get('task', 'research')}
async def run_planner(inp):     return {'plan': ['search', 'analyse', 'write'], 'planner_done': True}
async def run_researcher(inp):  return {'sources': ['source_a', 'source_b'], 'researcher_done': True}
async def run_analyst(inp):     return {'insights': ['insight_1', 'insight_2'], 'analyst_done': True}
async def run_writer(inp):      return {'draft': 'Draft report content...', 'writer_done': True}
async def run_critic(inp):      return {'score': 0.88, 'feedback': 'Good, minor revisions needed.', 'critic_done': True}
async def run_arxiv(inp):       return {'papers': ['paper_1', 'paper_2'], 'arxiv_done': True}
async def run_stats(inp):       return {'p_value': 0.03, 'significant': True, 'stats_done': True}

# Map agent names → Step objects
step_map = {
    'main_coordinator': Step('coordinator', run_coordinator),
    'research_planner': Step('planner',     run_planner),
    'web_researcher':   Step('researcher',  run_researcher),
    'data_analyst':     Step('analyst',     run_analyst),
    'report_writer':    Step('writer',      run_writer),
    'quality_critic':   Step('critic',      run_critic),
    'arxiv_specialist': Step('arxiv',       run_arxiv),
    'stats_specialist': Step('stats',       run_stats),
}

# Create and populate the HierarchicalPipeline
hp = HierarchicalPipeline()
hp.embed(hierarchy, step_map)

print(f'HierarchicalPipeline created')
print()

# flatten() — depth-first list of all steps
flat_steps = hp.flatten()
print(f'Flattened steps ({len(flat_steps)}):')
for step in flat_steps:
    print(f'  {step.name!r}')
print()

# describe() — nested structure dict
desc = hp.describe()
print('Pipeline description (nested dict):')
print(json.dumps(desc, indent=2))


## Section 10 — Running a HierarchicalPipeline

`await hp.run(initial_input)` executes the pipeline.  The default execution strategy
is **breadth-first**: the root step runs first, then all level-1 children run in
parallel (using `asyncio.gather`), then all level-2 children run in parallel, and
so on.

Each step receives the original `initial_input` merged with the accumulated outputs
from all ancestor steps on its path from the root.

`hp.run()` returns a dict mapping `agent_name → StepResult`.


In [ ]:
results = await hp.run({'task': 'survey of AI orchestration frameworks', 'depth': 2})

print(f'run() returned {len(results)} results (one per agent)')
print()

for agent_name, step_result in results.items():
    status = 'OK' if step_result.success else 'FAIL'
    print(f'  [{status}]  {agent_name!r:<22}  step={step_result.step_name!r:<14}  '
          f'duration={step_result.duration_ms:.3f}ms')
    if not step_result.success:
        print(f'         error: {step_result.error}')

print()

# Show what the critic received and returned
critic_result = results.get('quality_critic')
if critic_result:
    print('Critic step result:')
    print(f'  output : {critic_result.output}')
print()

# Show researcher's sub-worker (arxiv_specialist)
arxiv_result = results.get('arxiv_specialist')
if arxiv_result:
    print('arxiv_specialist step result:')
    print(f'  output : {arxiv_result.output}')


## Section 11 — StepKind Taxonomy

`StepKind` is an enum that **annotates** the semantic purpose of a step.  It does not
change execution behaviour — it is metadata for tooling, visualisation, and documentation.

| StepKind | Typical use |
|----------|-------------|
| `TRANSFORM` | Map input to a different format or enrich it |
| `FILTER` | Remove items that do not meet a condition |
| `AGGREGATE` | Combine multiple inputs into a summary |
| `BRANCH` | Conditional routing |
| `LOOP` | Iterative repetition |

You can attach `StepKind` to any `TypedStep` (or store it as metadata on a plain
`Step`) and use it later when inspecting pipeline structure.


In [ ]:
from multigen.hierarchy import StepKind, TypedStep, ValidationError

# Build a set of typed steps with explicit StepKind annotations

RECORDS_SCHEMA = {
    'type': 'object',
    'required': ['records'],
    'properties': {
        'records': {'type': 'array'},
    },
    'additionalProperties': True,
}

async def transform_fn(inp):
    return {'records': [{**r, 'transformed': True} for r in inp.get('records', [])]}

async def filter_fn(inp):
    return {'records': [r for r in inp.get('records', []) if r.get('score', 0) >= 0.5]}

async def aggregate_fn(inp):
    records = inp.get('records', [])
    return {'count': len(records), 'avg_score': sum(r.get('score', 0) for r in records) / max(len(records), 1)}

async def branch_fn(inp):
    return {'routed_to': 'high' if inp.get('count', 0) > 2 else 'low'}

async def loop_fn(inp):
    return {'iterations_simulated': inp.get('target_count', 3), 'loop_done': True}

typed_steps = [
    TypedStep('transform_step', transform_fn,  RECORDS_SCHEMA, RECORDS_SCHEMA, kind=StepKind.TRANSFORM),
    TypedStep('filter_step',    filter_fn,     RECORDS_SCHEMA, RECORDS_SCHEMA, kind=StepKind.FILTER),
    TypedStep('aggregate_step', aggregate_fn,  RECORDS_SCHEMA, {'type':'object','required':['count','avg_score'],'properties':{'count':{'type':'integer'},'avg_score':{'type':'number'}}}, kind=StepKind.AGGREGATE),
    TypedStep('branch_step',    branch_fn,     {'type':'object','additionalProperties':True}, {'type':'object','required':['routed_to'],'properties':{'routed_to':{'type':'string'}}}, kind=StepKind.BRANCH),
    TypedStep('loop_step',      loop_fn,       {'type':'object','additionalProperties':True}, {'type':'object','additionalProperties':True}, kind=StepKind.LOOP),
]

print('TypedStep objects with StepKind annotations:')
print(f'  {"name":<18}  {"kind":<12}  input_required')
for ts in typed_steps:
    required = ts.input_schema.get('required', [])
    print(f'  {ts.name!r:<18}  {ts.kind.value!r:<12}  {required}')
print()

# Run each step with sample input and show the kind in the output summary
sample_records = {'records': [{'id': 1, 'score': 0.9}, {'id': 2, 'score': 0.3}, {'id': 3, 'score': 0.7}]}

for ts in typed_steps:
    inp = {**sample_records, 'count': 2, 'target_count': 3}
    result = await ts.run(inp)
    print(f'[{ts.kind.value:<10}] {ts.name!r:<18}  success={result.success}  output={result.output}')


## Section 12 — Real-World Example: Research Pipeline Hierarchy

We now assemble a complete, realistic multi-agent research pipeline:

```
main_coordinator
├── research_planner
├── web_researcher
│   └── arxiv_specialist
├── data_analyst
│   └── stats_specialist
├── report_writer
└── quality_critic
```

Each agent is backed by a `TypedStep` with an appropriate `StepKind`.  The pipeline:

1. Coordinator decomposes the research goal into sub-tasks.
2. Planner orders the sub-tasks.
3. Researcher fetches sources; arxiv_specialist digs into papers.
4. Analyst extracts insights; stats_specialist runs significance tests.
5. Writer drafts the report.
6. Critic scores the draft and provides feedback.

All execution is in-memory with mock functions.


In [ ]:
from multigen.hierarchy import (
    AgentRole, AgentSpec, AgentHierarchy, StepKind, TypedStep, HierarchicalPipeline, ValidationError
)
from multigen.steps import Step

# ── Agent specs ───────────────────────────────────────────────────────────────

specs = {
    'coordinator': AgentSpec('coordinator',        AgentRole.COORDINATOR, ['task_dispatch', 'routing'],              '3.0.0'),
    'planner':     AgentSpec('planner',            AgentRole.PLANNER,     ['decomposition', 'ordering'],             '1.2.0'),
    'researcher':  AgentSpec('researcher',         AgentRole.WORKER,      ['web_search', 'summarisation'],           '2.0.0'),
    'arxiv':       AgentSpec('arxiv_specialist',   AgentRole.WORKER,      ['arxiv_api', 'paper_parsing'],            '1.0.0'),
    'analyst':     AgentSpec('analyst',            AgentRole.WORKER,      ['statistics', 'insight_extraction'],      '1.5.0'),
    'stats':       AgentSpec('stats_specialist',   AgentRole.WORKER,      ['regression', 'significance_testing'],    '1.0.0'),
    'writer':      AgentSpec('writer',             AgentRole.WORKER,      ['markdown', 'citations'],                 '1.1.0'),
    'critic':      AgentSpec('critic',             AgentRole.CRITIC,      ['fact_check', 'completeness', 'style'],   '2.0.0'),
}

# ── Build hierarchy ───────────────────────────────────────────────────────────

research_hierarchy = AgentHierarchy()
research_hierarchy.set_root(specs['coordinator'])

for child_key in ['planner', 'researcher', 'analyst', 'writer', 'critic']:
    research_hierarchy.add_child('coordinator', specs[child_key])

research_hierarchy.add_child('researcher', specs['arxiv'])
research_hierarchy.add_child('analyst',    specs['stats'])

print('Research pipeline hierarchy:')
research_hierarchy.visualize()
print()


In [ ]:
GENERIC_SCHEMA = {'type': 'object', 'additionalProperties': True}

# ── TypedStep implementations ─────────────────────────────────────────────────

async def coordinator_fn(inp):
    goal = inp.get('goal', 'unknown')
    return {
        'subtasks':   ['literature_review', 'data_analysis', 'synthesis', 'writing'],
        'goal':       goal,
        'priority':   'high',
        'coord_done': True,
    }

async def planner_fn(inp):
    subtasks = inp.get('subtasks', [])
    return {
        'execution_order': subtasks,
        'estimated_steps': len(subtasks),
        'plan_done':        True,
    }

async def researcher_fn(inp):
    goal = inp.get('goal', 'unknown')
    return {
        'sources':          [f'source_{i}' for i in range(1, 4)],
        'query_used':       goal,
        'researcher_done':  True,
    }

async def arxiv_fn(inp):
    return {
        'papers':       ['paper_A', 'paper_B', 'paper_C'],
        'total_found':  3,
        'arxiv_done':   True,
    }

async def analyst_fn(inp):
    sources = inp.get('sources', [])
    return {
        'insights':       [f'insight from {s}' for s in sources[:2]],
        'confidence':     0.87,
        'analyst_done':   True,
    }

async def stats_fn(inp):
    return {
        'test':       'two-sample t-test',
        'p_value':    0.021,
        'significant': True,
        'stats_done':  True,
    }

async def writer_fn(inp):
    insights  = inp.get('insights', [])
    n_sources = len(inp.get('sources', []))
    return {
        'draft':        f'## Research Report\n\n{chr(10).join(insights)}\n\nBased on {n_sources} source(s).',
        'word_count':   120,
        'writer_done':  True,
    }

async def critic_fn(inp):
    draft   = inp.get('draft', '')
    sources = inp.get('sources', [])
    score   = min(0.6 + len(sources) * 0.05, 1.0)
    return {
        'score':        round(score, 3),
        'passed':       score >= 0.7,
        'feedback':     'Well-structured. Add more quantitative evidence.' if score < 0.9 else 'Excellent.',
        'critic_done':  True,
    }

# ── Map specs → TypedSteps ────────────────────────────────────────────────────

research_step_map = {
    'coordinator':      TypedStep('coordinator', coordinator_fn, GENERIC_SCHEMA, GENERIC_SCHEMA, StepKind.TRANSFORM),
    'planner':          TypedStep('planner',     planner_fn,     GENERIC_SCHEMA, GENERIC_SCHEMA, StepKind.TRANSFORM),
    'researcher':       TypedStep('researcher',  researcher_fn,  GENERIC_SCHEMA, GENERIC_SCHEMA, StepKind.TRANSFORM),
    'arxiv_specialist': TypedStep('arxiv',       arxiv_fn,       GENERIC_SCHEMA, GENERIC_SCHEMA, StepKind.AGGREGATE),
    'analyst':          TypedStep('analyst',     analyst_fn,     GENERIC_SCHEMA, GENERIC_SCHEMA, StepKind.AGGREGATE),
    'stats_specialist': TypedStep('stats',       stats_fn,       GENERIC_SCHEMA, GENERIC_SCHEMA, StepKind.AGGREGATE),
    'writer':           TypedStep('writer',      writer_fn,      GENERIC_SCHEMA, GENERIC_SCHEMA, StepKind.TRANSFORM),
    'critic':           TypedStep('critic',      critic_fn,      GENERIC_SCHEMA, GENERIC_SCHEMA, StepKind.FILTER),
}

# ── Build and run HierarchicalPipeline ───────────────────────────────────────

research_pipeline = HierarchicalPipeline()
research_pipeline.embed(research_hierarchy, research_step_map)

print('Flattened execution order:')
for step in research_pipeline.flatten():
    print(f'  {step.name!r}')
print()

results = await research_pipeline.run({'goal': 'Impact of LLM agents on software engineering productivity'})

print('Pipeline execution results:')
print(f'  {"agent":<22}  {"step":<14}  {"status":<8}  key outputs')
print(f'  {"-"*22}  {"-"*14}  {"-"*8}  {"-"*30}')

for agent_name, sr in results.items():
    status = 'OK' if sr.success else 'FAIL'
    keys   = list(sr.output.keys()) if sr.output else []
    print(f'  {agent_name!r:<22}  {sr.step_name!r:<14}  {status:<8}  {keys}')

print()

# Final report quality gate
critic_result = results.get('critic')
if critic_result and critic_result.success:
    cr = critic_result.output
    gate = 'PASSED' if cr.get('passed') else 'FAILED'
    print(f'Quality gate: {gate}  (score={cr["score"]}, feedback={cr["feedback"]!r})')
print()

# Show the generated draft
writer_result = results.get('writer')
if writer_result and writer_result.success:
    print('Generated draft:')
    print(writer_result.output.get('draft', ''))
    print(f'Word count: {writer_result.output.get("word_count")}')


## Summary

### Hierarchical structures API at a glance

**Agent modelling**

| Class | Description |
|-------|-------------|
| `AgentRole` | Enum: `COORDINATOR`, `WORKER`, `CRITIC`, `PLANNER`, `EXECUTOR` |
| `AgentSpec` | Descriptor: name, role, capabilities, version, metadata |
| `AgentGroup` | Mutable collection; add/remove/filter by role |
| `AgentHierarchy` | Tree of AgentSpec; set_root, add_child, visualize, find, navigate |

**Typed pipeline components**

| Class | Description |
|-------|-------------|
| `StepKind` | Enum: `TRANSFORM`, `FILTER`, `AGGREGATE`, `BRANCH`, `LOOP` |
| `TypedStep` | Step + JSON Schema input/output + validate_input/validate_output |
| `ValidationError` | Raised when input or output fails schema validation |
| `HierarchicalPipeline` | Pipeline tree; embed(hierarchy, step_map), flatten, describe, run |

### Typical workflow

```
1. Define AgentSpec objects with roles and capabilities
2. Build an AgentHierarchy: set_root, add_child at each level
3. Write async functions for each agent
4. Wrap functions as TypedStep with input/output schemas and StepKind
5. hp = HierarchicalPipeline(); hp.embed(hierarchy, step_map)
6. results = await hp.run({'goal': '...'})
7. Inspect results[agent_name].output for each agent's contribution
```

### Next steps

- **Notebook 23**: Step composition operators — `>>`, `|`, `&`, `BranchStep`, `LoopStep`.
- **Notebook 20**: Time-travel debugging — record and replay pipeline executions.
- Combine `AgentHierarchy` with `WorkflowDebugger` to record a snapshot per agent
  in the hierarchy and replay from any node.
